# 03 — Iterators & Generators
**References:** Ramalho *Fluent Python* Ch. 17 · Beazley *Python Cookbook* Ch. 4 · PEP 255, 342, 380

## Narrative thread
```
Iterable vs Iterator -> Iterator protocol -> Generator functions -> yield from -> Generator pipeline -> itertools
```

## 1. The iterator protocol

An **iterable** has `__iter__()` that returns an **iterator**. An **iterator** has `__next__()` that returns the next item or raises `StopIteration`.

```
for x in obj:       ──>  it = iter(obj)   ──>  obj.__iter__()
    ...                  while True:
                             x = next(it) ──>  it.__next__()
                             ...               # raises StopIteration to end
```

Every iterator is also iterable — its `__iter__` returns `self`. This makes iterators one-pass (exhausted after one full traversal).

In [ ]:
import itertools
import operator
from typing import Generator, Iterator, Iterable

# ── Manual iterator: Fibonacci sequence ──────────────────────────────────
class FibIterator:
    """Infinite Fibonacci iterator. Shows the iterator protocol explicitly."""
    def __init__(self):
        self._a, self._b = 0, 1

    def __iter__(self):    # makes it usable in for loops
        return self

    def __next__(self) -> int:
        value       = self._a
        self._a, self._b = self._b, self._a + self._b
        return value

fib_it = FibIterator()
print('First 10 Fibonacci:', [next(fib_it) for _ in range(10)])
print('Iterators are one-pass — already advanced to 11th element')
print('Next value:', next(fib_it))

print()
# ── iter() with sentinel ─────────────────────────────────────────────────
import io, struct
data   = bytes(range(32))
stream = io.BytesIO(data)
CHUNK  = 4

# iter(callable, sentinel) — call callable until it returns sentinel
chunks = list(iter(lambda: stream.read(CHUNK), b''))
print(f'Read {len(chunks)} chunks of {CHUNK} bytes: {[c.hex() for c in chunks[:3]]}...')

print()
# ── Separation of concerns: iterable vs iterator ──────────────────────────
class Range:
    """Re-iterable (unlike an iterator): creates a new iterator each time."""
    def __init__(self, start, stop):
        self.start, self.stop = start, stop

    def __iter__(self) -> Iterator[int]:  # returns a NEW iterator each call
        current = self.start
        while current < self.stop:
            yield current          # using a generator here is idiomatic
            current += 1

r = Range(0, 5)
print('First pass: ', list(r))
print('Second pass:', list(r))  # works because Range is iterable, not an iterator

## 2. Generator functions

A **generator function** contains `yield` and returns a **generator object** (which is both an iterator and an iterable). Execution is suspended at each `yield` and resumed on `next()`.

**Generator protocol:**
- `next(g)` — resume execution to next `yield` or raise `StopIteration`
- `g.send(value)` — resume, and the current `yield` expression evaluates to `value`
- `g.throw(exc)` — raise `exc` at the `yield` point
- `g.close()` — raise `GeneratorExit` at the `yield` point

In [ ]:
# ── Generator as a coroutine (send/throw) ────────────────────────────────
def running_average() -> Generator[float, float, str]:
    """Coroutine that computes running average of sent values."""
    total = 0.0
    count = 0
    average = None
    while True:
        term = yield average       # pause here; receive next value via .send()
        if term is None:
            break
        total += term
        count += 1
        average = total / count
    return f'Total={total}, Count={count}'

coro = running_average()
next(coro)                     # prime the coroutine to the first yield
for v in [10, 20, 30, 40]:
    avg = coro.send(v)
    print(f'  sent {v:>3} -> running avg = {avg:.2f}')

try:
    coro.send(None)            # signal end; triggers return
except StopIteration as e:
    print(f'  Final: {e.value}')

print()
# ── yield from: delegation ────────────────────────────────────────────────
def chain(*iterables):
    """Like itertools.chain — delegates to sub-iterators via yield from."""
    for it in iterables:
        yield from it            # passes send/throw/close through to sub-gen

print('yield from chain:', list(chain([1,2,3], 'ABC', range(5,8))))

# yield from also propagates return values from sub-generators
def sub():
    yield 1
    yield 2
    return 'sub done'

def delegator():
    result = yield from sub()    # result captures sub()'s return value
    print(f'  sub returned: {result!r}')
    yield 3

print('yield from return value propagation:')
print(list(delegator()))

## 3. Generator pipelines and `itertools`

In [ ]:
# ── Generator pipeline: lazy, memory-efficient data processing ────────────
import random

def generate_events(n: int) -> Iterator[dict]:
    """Simulate a stream of events."""
    kinds = ['click', 'view', 'purchase', 'error']
    for i in range(n):
        yield {'id': i, 'kind': random.choice(kinds), 'value': random.randint(1, 100)}

def filter_kind(events: Iterable[dict], kind: str) -> Iterator[dict]:
    return (e for e in events if e['kind'] == kind)

def extract_value(events: Iterable[dict]) -> Iterator[int]:
    return (e['value'] for e in events)

def running_sum(values: Iterable[int]) -> Iterator[float]:
    total = 0
    for v in values:
        total += v
        yield total

# Pipeline — nothing evaluated until we consume the final iterator
random.seed(42)
events   = generate_events(1000)
clicks   = filter_kind(events, 'purchase')
values   = extract_value(clicks)
cumsum   = running_sum(values)
result   = list(itertools.islice(cumsum, 10))  # take first 10 cumulative sums
print('Purchase value cumulative sum (first 10):', result)

print()
# ── itertools showcase ────────────────────────────────────────────────────
print('itertools.islice      :', list(itertools.islice(range(100), 5, 15, 2)))
print('itertools.takewhile   :', list(itertools.takewhile(lambda x: x < 5, range(10))))
print('itertools.dropwhile   :', list(itertools.dropwhile(lambda x: x < 5, range(10))))
print('itertools.pairwise    :', list(itertools.pairwise('ABCDE')))
print('itertools.groupby     :', [(k, list(g)) for k, g in itertools.groupby('AAABBBCCCA')])
print('itertools.accumulate  :', list(itertools.accumulate([1,2,3,4,5], operator.mul)))
print('itertools.batched(3.12):', list(itertools.batched(range(10), 3)))

# Combinatorics
print('\ncombinations(3,2) :', list(itertools.combinations('ABC', 2)))
print('permutations(3,2) :', list(itertools.permutations('ABC', 2)))
print('product           :', list(itertools.product([0,1], repeat=3))[:4], '...')

## 4. Key takeaways

| Concept | Rule |
|---|---|
| Iterable | Has `__iter__()` — can be iterated multiple times |
| Iterator | Has `__next__()` — one-pass, stateful |
| Generator | Lazily produces values; suspends at `yield` |
| `yield from` | Delegates to sub-iterator; propagates `send`/`throw`/return |
| Pipeline | Chain generators for lazy, memory-efficient ETL |
| `itertools` | Composable, memory-efficient building blocks |
| `iter(fn, sentinel)` | Turn any callable into an iterator |

**Next:** notebook 04 — context managers: `__enter__`/`__exit__` and `contextlib`.